# Soft Actor-Critic-Based Semantic-Aware Power and Time Allocation for Hybrid NOMA-OFDMA Networks

**Project goal:** Given a configurable wireless system, a Soft Actor-Critic (SAC) agent learns how to allocate the available **transmit power** and **transmission time** among users while the simulator models OFDMA subcarriers and NOMA sharing.

### User-provided system inputs
1. Number of users
2. Number of OFDMA subcarriers
3. Total system bandwidth
4. Maximum base-station transmit-power budget
5. Frame/transmission duration

The simulator automatically generates per-user conditions such as position, distance, channel gain/fading, data demand, semantic importance, and QoS target. These are observations of the wireless environment—not values the user has to type manually.

> **Modeling note:** semantic communication is represented by an explicit **semantic-utility proxy** (importance-weighted information utility). This notebook does not claim to contain a neural semantic encoder/decoder or a real semantic dataset.

In [ ]:
# ============================================================
# CELL 1 — IMPORTS AND REPRODUCIBILITY
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
from collections import deque

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device  :", DEVICE)


## 2. Enter the network configuration

These are the **only five project inputs**. The remaining user/channel characteristics are generated by the simulator.

In [ ]:
# ============================================================
# CELL 2 — FIVE USER INPUTS
# ============================================================

def ask_int(prompt, default, minimum):
    raw = input(f"{prompt} [{default}]: ").strip()
    value = default if raw == "" else int(raw)
    if value < minimum:
        raise ValueError(f"Value must be >= {minimum}")
    return value

def ask_float(prompt, default, minimum):
    raw = input(f"{prompt} [{default}]: ").strip()
    value = default if raw == "" else float(raw)
    if value <= minimum:
        raise ValueError(f"Value must be > {minimum}")
    return value

print("ENTER SYSTEM PARAMETERS (press Enter to use the default)")
print("-------------------------------------------------------")
NUM_USERS = ask_int("Number of users", 20, 2)
NUM_SUBCARRIERS = ask_int("Number of OFDMA subcarriers", 5, 1)
TOTAL_BANDWIDTH_MHZ = ask_float("Total bandwidth (MHz)", 10.0, 0.0)
MAX_POWER_W = ask_float("Maximum transmit-power budget (W)", 10.0, 0.0)
FRAME_TIME_S = ask_float("Frame/transmission duration (s)", 1.0, 0.0)

TOTAL_BANDWIDTH_HZ = TOTAL_BANDWIDTH_MHZ * 1e6
SUBCARRIER_BW_HZ = TOTAL_BANDWIDTH_HZ / NUM_SUBCARRIERS

# Fixed simulation assumptions / physical-model parameters.
CARRIER_FREQUENCY_HZ = 2.4e9
NOISE_DENSITY_W_HZ = 4e-21
PATH_LOSS_EXPONENT = 3.0
AREA_SIZE_M = 100.0
MIN_DISTANCE_M = 5.0
MAX_EPISODES = 1200
BATCH_SIZE = 128
WARMUP_STEPS = 300
EVAL_EPISODES = 200

print("\nCONFIGURATION")
print(f"Users                 : {NUM_USERS}")
print(f"OFDMA subcarriers     : {NUM_SUBCARRIERS}")
print(f"Total bandwidth       : {TOTAL_BANDWIDTH_MHZ:.2f} MHz")
print(f"Bandwidth/subcarrier  : {SUBCARRIER_BW_HZ/1e6:.4f} MHz")
print(f"Maximum power budget  : {MAX_POWER_W:.2f} W")
print(f"Frame duration        : {FRAME_TIME_S:.4f} s")


In [ ]:
# ============================================================
# CELL 3 — SIMULATION MODEL AND USER REQUIREMENTS
# ============================================================
BS_X, BS_Y = AREA_SIZE_M / 2, AREA_SIZE_M / 2

# Per-user semantic importance and traffic/QoS requirements are generated
# automatically. A real deployment would obtain these from application data.
def generate_user_profile(n):
    # 50% semantic traffic, 50% conventional/bit traffic.
    is_semantic = np.random.rand(n) < 0.5
    semantic_importance = np.where(
        is_semantic,
        np.random.uniform(0.4, 1.0, n),
        0.0
    )
    # Target rate in Mbps. Semantic users can have moderate/high information demand.
    target_rate_mbps = np.random.uniform(0.5, 3.0, n)
    return is_semantic, semantic_importance, target_rate_mbps

is_semantic, semantic_importance, target_rate_mbps = generate_user_profile(NUM_USERS)

# User locations are generated inside the service area.
user_x = np.random.uniform(0, AREA_SIZE_M, NUM_USERS)
user_y = np.random.uniform(0, AREA_SIZE_M, NUM_USERS)
distance = np.maximum(
    np.sqrt((user_x - BS_X)**2 + (user_y - BS_Y)**2),
    MIN_DISTANCE_M
)

users = pd.DataFrame({
    "User_ID": np.arange(1, NUM_USERS + 1),
    "Communication_Type": np.where(is_semantic, "Semantic", "Bit"),
    "Semantic_Importance": semantic_importance,
    "Target_Rate_Mbps": target_rate_mbps,
    "X_m": user_x,
    "Y_m": user_y,
    "Distance_m": distance,
})

print(users.head(min(10, NUM_USERS)).round(4).to_string(index=False))
print("\nTotal users:", NUM_USERS)


In [ ]:
# ============================================================
# CELL 4 — CHANNEL MODEL AND HYBRID NOMA-OFDMA SCHEDULER
# ============================================================

def generate_channel():
    # Random user movement/channel realization for one decision frame.
    x = np.random.uniform(0, AREA_SIZE_M, NUM_USERS)
    y = np.random.uniform(0, AREA_SIZE_M, NUM_USERS)
    d = np.maximum(np.sqrt((x-BS_X)**2 + (y-BS_Y)**2), MIN_DISTANCE_M)

    # Rayleigh power fading and distance-based path loss.
    fading_power = np.random.exponential(scale=1.0, size=NUM_USERS)
    gain = fading_power / (d ** PATH_LOSS_EXPONENT)
    gain = np.maximum(gain, 1e-15)

    # Channel-quality proxy (higher is better). This is also used as a
    # normalized observation for SAC.
    quality = gain / np.max(gain)
    return x, y, d, gain, quality


def assign_ofdma_noma(gain):
    """Hybrid scheduler: assign users to OFDMA subcarriers, then allow
    users on the same subcarrier to share it using NOMA.

    Users are sorted by channel quality and distributed across subcarriers.
    Within each subcarrier, users are ordered from weaker to stronger channel.
    This keeps the resource assignment deterministic while SAC learns power/time.
    """
    order = np.argsort(gain)  # weak -> strong
    groups = [[] for _ in range(NUM_SUBCARRIERS)]
    for i, u in enumerate(order):
        groups[i % NUM_SUBCARRIERS].append(int(u))
    return groups


def noma_sinr_for_groups(gain, power, groups):
    sinr = np.zeros(NUM_USERS, dtype=np.float64)
    for group in groups:
        # Weak-to-strong order for a simple downlink NOMA model.
        group_sorted = sorted(group, key=lambda u: gain[u])
        for pos, u in enumerate(group_sorted):
            signal = power[u] * gain[u]
            # Stronger users are treated as residual intra-cluster interference
            # for this simplified SIC model.
            interference = sum(power[v] * gain[u] for v in group_sorted[pos+1:])
            sinr[u] = signal / (interference + NOISE_POWER_W + 1e-30)
    return sinr

# Thermal-noise approximation over the total system bandwidth.
NOISE_POWER_W = NOISE_DENSITY_W_HZ * TOTAL_BANDWIDTH_HZ

# Show one generated network snapshot.
x, y, d, g, q = generate_channel()
groups = assign_ofdma_noma(g)
users["Example_Channel_Gain"] = g
users["Example_Channel_Quality"] = q
users["OFDMA_Subcarrier"] = 0
for k, group in enumerate(groups):
    for u in group:
        users.loc[u, "OFDMA_Subcarrier"] = k + 1

print("OFDMA/NOMA groups for the example channel:")
for k, group in enumerate(groups, 1):
    print(f"  Subcarrier {k}: " + ", ".join(f"U{u+1}" for u in group))


In [ ]:
# ============================================================
# CELL 5 — VISUALIZE USERS AND CHANNEL QUALITY
# ============================================================
plt.figure(figsize=(7, 6))
plt.scatter(user_x, user_y, s=70, label="Users")
plt.scatter(BS_X, BS_Y, marker="^", s=180, label="Base Station")
for i in range(NUM_USERS):
    plt.text(user_x[i] + 1, user_y[i] + 1, f"U{i+1}", fontsize=8)
plt.xlabel("X position (m)")
plt.ylabel("Y position (m)")
plt.title("Simulated Wireless Network")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(9, 4))
plt.bar(np.arange(NUM_USERS) + 1, q)
plt.xlabel("User")
plt.ylabel("Normalized channel quality")
plt.title("Example User Channel Conditions")
plt.grid(axis="y")
plt.show()


In [ ]:
# ============================================================
# CELL 6 — RESOURCE EVALUATION FUNCTION
# ============================================================
def evaluate_resources(gain, groups, power, time_fraction):
    # OFDMA bandwidth is split equally across subcarriers in this model.
    bw_per_subcarrier = SUBCARRIER_BW_HZ
    user_bw = np.zeros(NUM_USERS)
    for group in groups:
        for u in group:
            user_bw[u] = bw_per_subcarrier

    sinr = noma_sinr_for_groups(gain, power, groups)
    spectral_eff = np.log2(1.0 + sinr)
    rate_bps = user_bw * spectral_eff
    rate_mbps = rate_bps / 1e6

    # Time allocation is a fraction of the frame. Effective delivered
    # information rate is weighted by the user's allocated time.
    delivered_mbits = rate_mbps * (time_fraction * FRAME_TIME_S)

    # Semantic utility proxy: important semantic users contribute more when
    # their allocated channel/time resources deliver information.
    semantic_utility = float(np.sum(
        semantic_importance * delivered_mbits
    ))

    # QoS satisfaction: fraction of target-rate demand served in the frame.
    qos_ratio = np.minimum(rate_mbps / np.maximum(target_rate_mbps, 1e-6), 1.0)
    mean_qos = float(np.mean(qos_ratio))

    total_throughput = float(np.sum(rate_mbps * time_fraction))
    used_power = float(np.sum(power))
    energy_efficiency = total_throughput / max(used_power, 1e-9)

    # Fairness of time-weighted achieved rate (Jain's fairness index).
    x = np.maximum(rate_mbps * time_fraction, 0.0)
    fairness = float((x.sum()**2) / (NUM_USERS * np.sum(x**2) + 1e-12))

    # Resource utilization penalties keep allocations inside meaningful ranges.
    power_utilization = used_power / MAX_POWER_W
    time_sum = float(np.sum(time_fraction))

    return {
        "sinr": sinr,
        "rate_mbps": rate_mbps,
        "delivered_mbits": delivered_mbits,
        "semantic_utility": semantic_utility,
        "qos_ratio": qos_ratio,
        "mean_qos": mean_qos,
        "throughput_mbps": total_throughput,
        "energy_efficiency": energy_efficiency,
        "fairness": fairness,
        "power_utilization": power_utilization,
        "time_sum": time_sum,
        "used_power": used_power,
        "user_bandwidth_hz": user_bw,
    }


def reward_from_metrics(m):
    # A transparent weighted objective. We normalize the terms to comparable
    # scales; the exact weights can be tuned for an experiment.
    throughput_score = m["throughput_mbps"] / max(NUM_USERS, 1)
    semantic_score = m["semantic_utility"] / max(NUM_USERS, 1)
    qos_score = m["mean_qos"]
    fairness_score = m["fairness"]
    energy_score = m["energy_efficiency"] / 10.0
    return float(
        0.30 * throughput_score +
        0.30 * semantic_score +
        0.15 * qos_score +
        0.15 * fairness_score +
        0.10 * energy_score
    )


In [ ]:
# ============================================================
# CELL 7 — EQUAL-ALLOCATION BASELINE
# ============================================================
def equal_allocation(gain, groups):
    power = np.full(NUM_USERS, MAX_POWER_W / NUM_USERS, dtype=np.float64)
    time_fraction = np.full(NUM_USERS, 1.0 / NUM_USERS, dtype=np.float64)
    return power, time_fraction


## 8. SAC environment

The SAC agent observes:
- normalized channel quality
- normalized distance
- semantic importance
- normalized target data rate
- current OFDMA subcarrier index

The action contains continuous logits for **power allocation** and **time allocation**, plus one scalar controlling how much of the available power budget is used.

The OFDMA subcarrier assignment is handled by the hybrid scheduler; users sharing a subcarrier form a NOMA cluster.

In [ ]:
# ============================================================
# CELL 8 — SAC ENVIRONMENT
# ============================================================
class HybridNOMAOFDMAEnv:
    def __init__(self):
        self.n = NUM_USERS
        self.state_dim = 5 * self.n
        self.action_dim = 2 * self.n + 1
        self.distance = None
        self.channel_gain = None
        self.channel_quality = None
        self.groups = None
        self.state = None

    def _build_state(self):
        g = self.channel_quality
        d = self.distance / (np.sqrt(2) * AREA_SIZE_M)
        s = semantic_importance
        r = target_rate_mbps / max(np.max(target_rate_mbps), 1e-9)
        sc = np.zeros(self.n, dtype=np.float32)
        for k, group in enumerate(self.groups):
            for u in group:
                sc[u] = k / max(NUM_SUBCARRIERS - 1, 1)
        return np.concatenate([g, d, s, r, sc]).astype(np.float32)

    def reset(self):
        _, _, self.distance, self.channel_gain, self.channel_quality = generate_channel()
        self.groups = assign_ofdma_noma(self.channel_gain)
        self.state = self._build_state()
        return self.state.copy()

    @staticmethod
    def _softmax_np(x):
        z = x - np.max(x)
        e = np.exp(z)
        return e / (np.sum(e) + 1e-12)

    def action_to_resources(self, action):
        action = np.asarray(action, dtype=np.float32)
        p_logits = action[:self.n]
        t_logits = action[self.n:2*self.n]
        power_fraction = self._softmax_np(p_logits)
        time_fraction = self._softmax_np(t_logits)

        # SAC decides the fraction of the available power budget to use.
        power_level = 0.05 + 0.95 * ((float(action[-1]) + 1.0) / 2.0)
        power_level = float(np.clip(power_level, 0.05, 1.0))
        total_power = power_level * MAX_POWER_W
        power = power_fraction * total_power
        return power, time_fraction, total_power

    def step(self, action):
        power, time_fraction, total_power = self.action_to_resources(action)
        metrics = evaluate_resources(self.channel_gain, self.groups, power, time_fraction)
        reward = reward_from_metrics(metrics)
        info = {**metrics, "total_power": total_power, "groups": self.groups}
        # One frame = one decision. A new episode gives a new channel realization.
        return self.state.copy(), reward, True, info


env = HybridNOMAOFDMAEnv()
state = env.reset()
print("State dimension :", env.state_dim)
print("Action dimension:", env.action_dim)


In [ ]:
# ============================================================
# CELL 9 — REPLAY BUFFER
# ============================================================
class ReplayBuffer:
    def __init__(self, capacity=50000):
        self.buffer = deque(maxlen=capacity)

    def add(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, ns, d = zip(*batch)
        return (
            torch.tensor(np.asarray(s), dtype=torch.float32, device=DEVICE),
            torch.tensor(np.asarray(a), dtype=torch.float32, device=DEVICE),
            torch.tensor(np.asarray(r), dtype=torch.float32, device=DEVICE).unsqueeze(1),
            torch.tensor(np.asarray(ns), dtype=torch.float32, device=DEVICE),
            torch.tensor(np.asarray(d), dtype=torch.float32, device=DEVICE).unsqueeze(1),
        )

    def __len__(self):
        return len(self.buffer)


In [ ]:
# ============================================================
# CELL 10 — SAC NETWORKS
# ============================================================
class MLP(nn.Module):
    def __init__(self, in_dim, out_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, out_dim)
        )
    def forward(self, x):
        return self.net(x)


class GaussianActor(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU()
        )
        self.mu = nn.Linear(hidden, action_dim)
        self.log_std = nn.Linear(hidden, action_dim)

    def forward(self, state):
        h = self.backbone(state)
        mu = self.mu(h)
        log_std = torch.clamp(self.log_std(h), -5, 2)
        return mu, log_std

    def sample(self, state):
        mu, log_std = self(state)
        std = log_std.exp()
        dist = torch.distributions.Normal(mu, std)
        z = dist.rsample()
        action = torch.tanh(z)
        # Tanh correction for the SAC log probability.
        log_prob = dist.log_prob(z) - torch.log(1 - action.pow(2) + 1e-6)
        log_prob = log_prob.sum(dim=1, keepdim=True)
        return action, log_prob

    def deterministic(self, state):
        mu, _ = self(state)
        return torch.tanh(mu)


class Critic(nn.Module):
    def __init__(self, state_dim, action_dim, hidden=128):
        super().__init__()
        self.q = nn.Sequential(
            nn.Linear(state_dim + action_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1)
        )
    def forward(self, state, action):
        return self.q(torch.cat([state, action], dim=1))


In [ ]:
# ============================================================
# CELL 11 — SOFT ACTOR-CRITIC AGENT
# ============================================================
class SACAgent:
    def __init__(self, state_dim, action_dim, gamma=0.99, tau=0.005, lr=3e-4):
        self.gamma = gamma
        self.tau = tau
        self.action_dim = action_dim

        self.actor = GaussianActor(state_dim, action_dim).to(DEVICE)
        self.q1 = Critic(state_dim, action_dim).to(DEVICE)
        self.q2 = Critic(state_dim, action_dim).to(DEVICE)
        self.tq1 = Critic(state_dim, action_dim).to(DEVICE)
        self.tq2 = Critic(state_dim, action_dim).to(DEVICE)
        self.tq1.load_state_dict(self.q1.state_dict())
        self.tq2.load_state_dict(self.q2.state_dict())

        self.actor_opt = optim.Adam(self.actor.parameters(), lr=lr)
        self.q1_opt = optim.Adam(self.q1.parameters(), lr=lr)
        self.q2_opt = optim.Adam(self.q2.parameters(), lr=lr)

        self.target_entropy = -float(action_dim)
        self.log_alpha = torch.zeros(1, requires_grad=True, device=DEVICE)
        self.alpha_opt = optim.Adam([self.log_alpha], lr=lr)

    @property
    def alpha(self):
        return self.log_alpha.exp()

    def select_action(self, state, evaluate=False):
        s = torch.tensor(state, dtype=torch.float32, device=DEVICE).unsqueeze(0)
        with torch.no_grad():
            a = self.actor.deterministic(s) if evaluate else self.actor.sample(s)[0]
        return a.cpu().numpy()[0]

    def update(self, replay, batch_size):
        state, action, reward, next_state, done = replay.sample(batch_size)
        with torch.no_grad():
            next_action, next_logp = self.actor.sample(next_state)
            target_q = torch.min(self.tq1(next_state, next_action), self.tq2(next_state, next_action))
            target = reward + self.gamma * (1 - done) * (target_q - self.alpha.detach() * next_logp)

        q1_loss = F.mse_loss(self.q1(state, action), target)
        q2_loss = F.mse_loss(self.q2(state, action), target)
        self.q1_opt.zero_grad(); q1_loss.backward(); self.q1_opt.step()
        self.q2_opt.zero_grad(); q2_loss.backward(); self.q2_opt.step()

        new_action, logp = self.actor.sample(state)
        new_q = torch.min(self.q1(state, new_action), self.q2(state, new_action))
        actor_loss = (self.alpha.detach() * logp - new_q).mean()
        self.actor_opt.zero_grad(); actor_loss.backward(); self.actor_opt.step()

        alpha_loss = -(self.log_alpha * (logp + self.target_entropy).detach()).mean()
        self.alpha_opt.zero_grad(); alpha_loss.backward(); self.alpha_opt.step()

        with torch.no_grad():
            for tp, p in zip(self.tq1.parameters(), self.q1.parameters()):
                tp.mul_(1-self.tau).add_(self.tau*p)
            for tp, p in zip(self.tq2.parameters(), self.q2.parameters()):
                tp.mul_(1-self.tau).add_(self.tau*p)

        return float(actor_loss.item()), float(q1_loss.item()+q2_loss.item())

agent = SACAgent(env.state_dim, env.action_dim)
replay = ReplayBuffer()
print("SAC agent ready.")


In [ ]:
# ============================================================
# CELL 12 — TRAIN SAC
# ============================================================
reward_history = []
throughput_history = []
semantic_history = []
qos_history = []
energy_history = []
loss_history = []

for episode in range(1, MAX_EPISODES + 1):
    state = env.reset()
    if episode <= WARMUP_STEPS:
        action = np.random.uniform(-1, 1, env.action_dim).astype(np.float32)
    else:
        action = agent.select_action(state, evaluate=False)

    next_state, reward, done, info = env.step(action)
    replay.add(state, action, reward, next_state, done)

    actor_loss = np.nan
    critic_loss = np.nan
    if len(replay) >= BATCH_SIZE:
        actor_loss, critic_loss = agent.update(replay, BATCH_SIZE)

    reward_history.append(reward)
    throughput_history.append(info["throughput_mbps"])
    semantic_history.append(info["semantic_utility"])
    qos_history.append(info["mean_qos"])
    energy_history.append(info["energy_efficiency"])
    loss_history.append(critic_loss)

    if episode % 100 == 0:
        print(
            f"Episode {episode:4d} | Reward {reward:8.4f} | "
            f"Throughput {info['throughput_mbps']:7.3f} Mbps | "
            f"Semantic {info['semantic_utility']:7.3f} | "
            f"QoS {info['mean_qos']:.3f} | Alpha {agent.alpha.item():.4f}"
        )

print("\nSAC training completed.")


In [ ]:
# ============================================================
# CELL 13 — TRAINING CURVES
# ============================================================
def moving_average(v, w=50):
    v = np.asarray(v, dtype=float)
    if len(v) < w:
        return v
    return np.convolve(v, np.ones(w)/w, mode="valid")

figs = [
    (reward_history, "Reward", "SAC Training Reward"),
    (throughput_history, "Throughput (Mbps)", "Throughput During Training"),
    (semantic_history, "Semantic Utility", "Semantic Utility During Training"),
    (qos_history, "Mean QoS Satisfaction", "QoS Satisfaction During Training"),
]
for values, ylabel, title in figs:
    plt.figure(figsize=(8,4))
    plt.plot(values, alpha=0.3, label="Episode")
    ma = moving_average(values)
    x = np.arange(len(ma)) + (49 if len(values) >= 50 else 0)
    plt.plot(x, ma, linewidth=2, label="50-episode average")
    plt.xlabel("Episode")
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid(True)
    plt.legend()
    plt.show()


In [ ]:
# ============================================================
# CELL 14 — SAC VS EQUAL-ALLOCATION BASELINE
# ============================================================
def evaluate_agent(agent, episodes=200):
    rows = []
    for _ in range(episodes):
        state = env.reset()
        action = agent.select_action(state, evaluate=True)
        _, reward, _, info = env.step(action)
        rows.append({
            "Reward": reward,
            "Throughput_Mbps": info["throughput_mbps"],
            "Semantic_Utility": info["semantic_utility"],
            "QoS": info["mean_qos"],
            "Fairness": info["fairness"],
            "Energy_Efficiency_Mbps_per_W": info["energy_efficiency"],
            "Power_W": info["used_power"],
        })
    return pd.DataFrame(rows)

sac_eval = evaluate_agent(agent, EVAL_EPISODES)

baseline_rows = []
for _ in range(EVAL_EPISODES):
    _, _, d, gain, _ = generate_channel()
    groups = assign_ofdma_noma(gain)
    power, time_fraction = equal_allocation(gain, groups)
    m = evaluate_resources(gain, groups, power, time_fraction)
    baseline_rows.append({
        "Reward": reward_from_metrics(m),
        "Throughput_Mbps": m["throughput_mbps"],
        "Semantic_Utility": m["semantic_utility"],
        "QoS": m["mean_qos"],
        "Fairness": m["fairness"],
        "Energy_Efficiency_Mbps_per_W": m["energy_efficiency"],
        "Power_W": m["used_power"],
    })
baseline_eval = pd.DataFrame(baseline_rows)

summary = pd.DataFrame({
    "Metric": ["Reward", "Throughput (Mbps)", "Semantic Utility", "QoS Satisfaction", "Fairness", "Energy Efficiency (Mbps/W)", "Used Power (W)"],
    "Equal Allocation": baseline_eval.mean(numeric_only=True).loc[["Reward","Throughput_Mbps","Semantic_Utility","QoS","Fairness","Energy_Efficiency_Mbps_per_W","Power_W"]].values,
    "SAC Allocation": sac_eval.mean(numeric_only=True).loc[["Reward","Throughput_Mbps","Semantic_Utility","QoS","Fairness","Energy_Efficiency_Mbps_per_W","Power_W"]].values,
})
print(summary.round(4).to_string(index=False))


In [ ]:
# ============================================================
# CELL 15 — FINAL LEARNED ALLOCATION FOR ONE NETWORK SNAPSHOT
# ============================================================
state = env.reset()
action = agent.select_action(state, evaluate=True)
power, time_fraction, total_power = env.action_to_resources(action)
_, reward, _, info = env.step(action)

result = pd.DataFrame({
    "User_ID": np.arange(1, NUM_USERS+1),
    "Type": np.where(is_semantic, "Semantic", "Bit"),
    "Semantic_Importance": semantic_importance,
    "Target_Rate_Mbps": target_rate_mbps,
    "Distance_m": env.distance,
    "Channel_Gain": env.channel_gain,
    "Subcarrier": 0,
    "Power_W": power,
    "Time_Allocation_s": time_fraction * FRAME_TIME_S,
    "SINR": info["sinr"],
    "SINR_dB": 10*np.log10(np.maximum(info["sinr"], 1e-12)),
    "Rate_Mbps": info["rate_mbps"],
    "QoS_Ratio": info["qos_ratio"],
})
for k, group in enumerate(info["groups"], 1):
    for u in group:
        result.loc[u, "Subcarrier"] = k

print("FINAL SAC RESOURCE ALLOCATION — ONE NETWORK REALIZATION")
print(result.round(4).to_string(index=False))
print("\nTotal power used:", round(total_power, 4), "W")
print("Total allocated time:", round(np.sum(time_fraction)*FRAME_TIME_S, 4), "s")
print("Reward:", round(reward, 4))


In [ ]:
# ============================================================
# CELL 16 — VISUAL COMPARISON
# ============================================================
metrics_to_plot = [
    ("Throughput_Mbps", "Throughput (Mbps)"),
    ("Semantic_Utility", "Semantic Utility"),
    ("QoS", "QoS Satisfaction"),
    ("Energy_Efficiency_Mbps_per_W", "Energy Efficiency (Mbps/W)"),
]
for col, label in metrics_to_plot:
    plt.figure(figsize=(7,4))
    means = [baseline_eval[col].mean(), sac_eval[col].mean()]
    plt.bar(["Equal Allocation", "SAC"], means)
    plt.ylabel(label)
    plt.title(f"Baseline vs SAC: {label}")
    plt.grid(axis="y")
    plt.show()


## 17. Final project interpretation

### What the notebook demonstrates
- The base station has a configurable total power budget, bandwidth, and frame duration.
- The network can contain an arbitrary number of users and OFDMA subcarriers selected at the start of a run.
- The simulator generates each user's distance/channel realization, semantic importance, and traffic requirement.
- OFDMA provides separate subcarrier resources; users assigned to the same subcarrier share it using a simplified NOMA/SIC model.
- SAC continuously outputs resource-allocation decisions for **power and time**.
- The reward balances throughput, semantic utility, QoS satisfaction, fairness, and energy efficiency.
- SAC is evaluated against an equal power/equal time baseline.

### Important limitation to state in a viva/report
This is a **simulation and reinforcement-learning resource-allocation model**. The semantic term is an importance-weighted utility proxy; there is no real semantic encoder/decoder, real radio hardware, or measured channel dataset in this notebook.